In [1]:
import pandas as pd
from tqdm import tqdm
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

# Limpeza da base do ENEM 2023

Este notebook prepara a base bruta para modelagem e analise. O fluxo faz a leitura do arquivo, remove colunas sem uso no recorte, filtra registros invalidos, renomeia variaveis socioeconomicas, cria a variavel `NU_MEDIA_GERAL` e exporta as bases finais para `bases_limpas`.

In [2]:
# Caminho do microdado bruto utilizado em todo o processo de limpeza.
csv_path = "../DADOS/MICRODADOS_ENEM_2023.csv"

# Carrega a base completa do ENEM 2023 com a codificacao original do arquivo.
base = pd.read_csv(csv_path,encoding='latin1', sep=';')

In [3]:
# Mantem uma copia da base original para aplicar as transformacoes sem perder o dado bruto.
base2 = base.copy()
# Visualizacao inicial para conferir a estrutura antes das remocoes.
base

,NU_INSCRICAO,NU_ANO,TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_NACIONALIDADE,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ESCOLA,...,Q016,Q017,Q018,Q019,Q020,Q021,Q022,Q023,Q024,Q025
0,210059085136,2023,14,M,2,1,1,1,17,1,...,C,C,B,B,A,B,B,A,A,B
1,210059527735,2023,12,M,2,1,0,1,16,1,...,B,A,B,B,A,A,C,A,D,B
2,210061103945,2023,6,F,1,1,1,1,0,1,...,B,A,A,B,A,A,A,A,A,B
3,210060214087,2023,2,F,1,3,1,2,0,2,...,A,A,A,B,A,A,D,A,A,B
4,210059980948,2023,3,F,1,3,1,2,0,2,...,A,A,A,B,A,A,B,A,A,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3933950,210061959676,2023,12,M,1,1,1,1,6,1,...,B,A,A,C,A,B,E,A,A,B
3933951,210061950911,2023,1,F,1,1,2,3,0,1,...,B,A,B,C,B,B,B,B,C,B
3933952,210061965966,2023,3,F,1,3,1,2,0,2,...,A,A,A,B,A,A,B,A,A,B
3933953,210061932304,2023,2,M,1,1,1,2,0,3,...,B,B,B,C,A,A,D,A,C,B


In [4]:
pd.options.display.max_columns = None
pd.options.display.max_rows = None

In [5]:
(base['NU_NOTA_MT'] == 0).sum()

np.int64(16638)

In [6]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3933955 entries, 0 to 3933954
Data columns (total 76 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   NU_INSCRICAO            int64  
 1   NU_ANO                  int64  
 2   TP_FAIXA_ETARIA         int64  
 3   TP_SEXO                 object 
 4   TP_ESTADO_CIVIL         int64  
 5   TP_COR_RACA             int64  
 6   TP_NACIONALIDADE        int64  
 7   TP_ST_CONCLUSAO         int64  
 8   TP_ANO_CONCLUIU         int64  
 9   TP_ESCOLA               int64  
 10  TP_ENSINO               float64
 11  IN_TREINEIRO            int64  
 12  CO_MUNICIPIO_ESC        float64
 13  NO_MUNICIPIO_ESC        object 
 14  CO_UF_ESC               float64
 15  SG_UF_ESC               object 
 16  TP_DEPENDENCIA_ADM_ESC  float64
 17  TP_LOCALIZACAO_ESC      float64
 18  TP_SIT_FUNC_ESC         float64
 19  CO_MUNICIPIO_PROVA      int64  
 20  NO_MUNICIPIO_PROVA      object 
 21  CO_UF_PROVA             int64  

In [ ]:
base2.columns

## DROP

In [7]:
linhas = base2.shape[0]
colunas = base2.shape[1]

### Sem utilidade

In [8]:
# Remove identificadores e colunas que nao entram na analise final.
base2 = base2.drop(columns=['NU_INSCRICAO','NU_ANO'])
base2 = base2.drop(columns=['CO_MUNICIPIO_PROVA','TP_NACIONALIDADE','CO_UF_PROVA','NO_MUNICIPIO_PROVA'])
base2 = base2.drop(columns=['TP_ESTADO_CIVIL','TP_COR_RACA','SG_UF_PROVA','TP_ST_CONCLUSAO','TP_ANO_CONCLUIU',
            'TP_ENSINO','TP_LINGUA','NU_NOTA_COMP1', 'NU_NOTA_COMP2', 'NU_NOTA_COMP3', 'NU_NOTA_COMP4','NU_NOTA_COMP5'])

In [9]:
final_total_linhas = base2.shape[0]
final_total_colunas = base2.shape[1]
print(f"Comecou: {linhas} finalizou: {final_total_linhas} retirei: {linhas - final_total_linhas} - {round((linhas -final_total_linhas) / linhas,2) }% ")
print(f"Comecou: {colunas} finalizou: {final_total_colunas} retirei: {colunas - final_total_colunas} - {round((colunas -final_total_colunas) / colunas,2)} %")
linhas = base2.shape[0]
colunas = base2.shape[1]

Comecou: 3933955 finalizou: 3933955 retirei: 0 - 0.0% 
Comecou: 76 finalizou: 58 retirei: 18 - 0.24 %


### Treineiros 

In [10]:
base2['IN_TREINEIRO'].value_counts()

IN_TREINEIRO
0    3313888
1     620067
Name: count, dtype: int64

In [11]:
# Mantem apenas participantes validos e remove a flag apos o filtro.
base2 = base2[base2['IN_TREINEIRO'] == 0]
base2 = base2.drop(columns=['IN_TREINEIRO'])

In [12]:
final_total_linhas = base2.shape[0]
final_total_colunas = base2.shape[1]
print(f"Comecou: {linhas} finalizou: {final_total_linhas} retirei: {linhas - final_total_linhas} - {round((linhas -final_total_linhas) / linhas,2) }% ")
print(f"Comecou: {colunas} finalizou: {final_total_colunas} retirei: {colunas - final_total_colunas} - {round((colunas -final_total_colunas) / colunas,2)} %")
linhas = base2.shape[0]
colunas = base2.shape[1]

Comecou: 3933955 finalizou: 3313888 retirei: 620067 - 0.16% 
Comecou: 58 finalizou: 57 retirei: 1 - 0.02 %


### Presenca na CN, CH, LC, MT

In [13]:
base[base['IN_TREINEIRO']==0]['TP_PRESENCA_MT'].value_counts()

TP_PRESENCA_MT
1    2175507
0    1136546
2       1835
Name: count, dtype: int64

In [14]:
# Mantem somente quem esteve presente em todas as provas objetivas.
base2 = base2[base2['TP_PRESENCA_CN'] == 1]
base2 = base2[base2['TP_PRESENCA_CH'] == 1]
base2 = base2[base2['TP_PRESENCA_LC'] == 1]
# Depois do filtro, as colunas de presenca deixam de agregar informacao.
base2 = base2.drop(columns=['TP_PRESENCA_CN','TP_PRESENCA_CH','TP_PRESENCA_LC','TP_PRESENCA_MT'])

In [15]:
final_total_linhas = base2.shape[0]
final_total_colunas = base2.shape[1]
print(f"Comecou: {linhas} finalizou: {final_total_linhas} retirei: {linhas - final_total_linhas} - {round((linhas -final_total_linhas) / linhas,2) }% ")
print(f"Comecou: {colunas} finalizou: {final_total_colunas} retirei: {colunas - final_total_colunas} - {round((colunas -final_total_colunas) / colunas,2)} %")
linhas = base2.shape[0]
colunas = base2.shape[1]

Comecou: 3313888 finalizou: 2166843 retirei: 1147045 - 0.35% 
Comecou: 57 finalizou: 53 retirei: 4 - 0.07 %


### Cor da prova

In [16]:
# O codigo da prova nao e necessario para o recorte final analisado.
base2 = base2.drop(columns=['CO_PROVA_CN','CO_PROVA_CH','CO_PROVA_LC','CO_PROVA_MT'])

In [17]:
final_total_linhas = base2.shape[0]
final_total_colunas = base2.shape[1]
print(f"Comecou: {linhas} finalizou: {final_total_linhas} retirei: {linhas - final_total_linhas} - {round((linhas -final_total_linhas) / linhas,2) }% ")
print(f"Comecou: {colunas} finalizou: {final_total_colunas} retirei: {colunas - final_total_colunas} - {round((colunas -final_total_colunas) / colunas,2)} %")
linhas = base2.shape[0]
colunas = base2.shape[1]

Comecou: 2166843 finalizou: 2166843 retirei: 0 - 0.0% 
Comecou: 53 finalizou: 49 retirei: 4 - 0.08 %


### Resposta e gabarito 

In [18]:
# Remove respostas marcadas e gabaritos para reduzir a dimensionalidade da base final.
base2 = base2.drop(columns=['TX_RESPOSTAS_CN','TX_RESPOSTAS_CH','TX_RESPOSTAS_LC','TX_RESPOSTAS_MT'])
base2 = base2.drop(columns=['TX_GABARITO_CN','TX_GABARITO_CH','TX_GABARITO_LC','TX_GABARITO_MT'])

In [19]:
final_total_linhas = base2.shape[0]
final_total_colunas = base2.shape[1]
print(f"Comecou: {linhas} finalizou: {final_total_linhas} retirei: {linhas - final_total_linhas} - {round((linhas -final_total_linhas) / linhas,2) }% ")
print(f"Comecou: {colunas} finalizou: {final_total_colunas} retirei: {colunas - final_total_colunas} - {round((colunas -final_total_colunas) / colunas,2)} %")
linhas = base2.shape[0]
colunas = base2.shape[1]

Comecou: 2166843 finalizou: 2166843 retirei: 0 - 0.0% 
Comecou: 49 finalizou: 41 retirei: 8 - 0.16 %


### Status redacao

In [20]:
# O status da redacao deixa de ser necessario apos os filtros de consistencia adotados.
base2 = base2.drop(columns=['TP_STATUS_REDACAO'])

In [21]:
final_total_linhas = base2.shape[0]
final_total_colunas = base2.shape[1]
print(f"Comecou: {linhas} finalizou: {final_total_linhas} retirei: {linhas - final_total_linhas} - {round((linhas -final_total_linhas) / linhas,2) }% ")
print(f"Comecou: {colunas} finalizou: {final_total_colunas} retirei: {colunas - final_total_colunas} - {round((colunas -final_total_colunas) / colunas,2)} %")
linhas = base2.shape[0]
colunas = base2.shape[1]

Comecou: 2166843 finalizou: 2166843 retirei: 0 - 0.0% 
Comecou: 41 finalizou: 40 retirei: 1 - 0.02 %


### Valores com mais de x% valores nulos 

In [22]:
import pandas as pd

def remover_colunas_com_muitos_nulos(df, x, proteger=("TP_DEPENDENCIA_ADM_ESC",)):
    """
    REMOVE COLUNAS COM MAIS DE X% DE VALORES NULOS, EXCETO AS COLUNAS PROTEGIDAS.

    PARÂMETROS:
        df: DATAFRAME ORIGINAL
        x : PERCENTUAL LIMITE DE VALORES NULOS (0–100)
        proteger: TUPLA/LISTA DE COLUNAS QUE NÃO DEVEM SER REMOVIDAS

    RETORNA:
        df_filtrado: DATAFRAME SEM AS COLUNAS REMOVIDAS
        colunas_removidas: LISTA DAS COLUNAS REMOVIDAS
    """
    limite = x / 100.0
    proporcao_nulos = df.isna().mean()

    # GARANTE QUE SÓ TENTAREMOS PROTEGER COLUNAS QUE EXISTEM NO DF
    proteger = [c for c in (proteger or []) if c in df.columns]

    # IDENTIFICA COLUNAS CANDIDATAS À REMOÇÃO (> LIMITE) E NÃO PROTEGIDAS
    remover_mask = (proporcao_nulos > limite) & (~proporcao_nulos.index.isin(proteger))
    colunas_removidas = proporcao_nulos.index[remover_mask].tolist()

    # MONTA COLUNAS A MANTER
    colunas_para_manter = [c for c in df.columns if c not in colunas_removidas]
    df_filtrado = df[colunas_para_manter].copy()

    # RELATÓRIO
    if proteger:
        print(f"COLUNAS PROTEGIDAS (NÃO REMOVIDAS INDEPENDENTE DE NULOS): {proteger}")
    print(f"\nCOLUNAS REMOVIDAS (MAIS DE {x}% DE NULOS):")
    for col in colunas_removidas:
        print(f" - {col}")
    if not colunas_removidas:
        print(" - NENHUMA")

    return df_filtrado, colunas_removidas


In [23]:
# Remove colunas com excesso de nulos, preservando variaveis essenciais para a segmentacao final.
base2, removidas = remover_colunas_com_muitos_nulos(
    base2, 50, proteger=("TP_DEPENDENCIA_ADM_ESC", "TP_ESCOLA")
)


COLUNAS PROTEGIDAS (NÃO REMOVIDAS INDEPENDENTE DE NULOS): ['TP_DEPENDENCIA_ADM_ESC', 'TP_ESCOLA']

COLUNAS REMOVIDAS (MAIS DE 50% DE NULOS):
 - CO_MUNICIPIO_ESC
 - NO_MUNICIPIO_ESC
 - CO_UF_ESC
 - SG_UF_ESC
 - TP_LOCALIZACAO_ESC
 - TP_SIT_FUNC_ESC


In [24]:
final_total_linhas = base2.shape[0]
final_total_colunas = base2.shape[1]
print(f"Comecou: {linhas} finalizou: {final_total_linhas} retirei: {linhas - final_total_linhas} - {round((linhas -final_total_linhas) / linhas,2) }% ")
print(f"Comecou: {colunas} finalizou: {final_total_colunas} retirei: {colunas - final_total_colunas} - {round((colunas -final_total_colunas) / colunas,2)} %")
linhas = base2.shape[0]
colunas = base2.shape[1]

Comecou: 2166843 finalizou: 2166843 retirei: 0 - 0.0% 
Comecou: 40 finalizou: 34 retirei: 6 - 0.15 %


In [25]:
# Renomeia as questoes socioeconomicas para nomes mais legiveis na analise.
base2.rename(columns = {'Q001':'Q01_Educacao_Pai'}, inplace = True)
base2.rename(columns = {'Q002':'Q02_Educacao_Mae'}, inplace = True)
base2.rename(columns = {'Q003':'Q03_Grupo_Ocupacao_Pai'}, inplace = True)
base2.rename(columns = {'Q004':'Q04_Grupo_Ocupacao_Mae'}, inplace = True)
base2.rename(columns = {'Q005':'Q05_Moradores/Residencia'}, inplace = True)
base2.rename(columns = {'Q006':'Q06_Renda_Familiar_Mensal'}, inplace = True)
base2.rename(columns = {'Q007':'Q07_Empregado_Domestico'}, inplace = True)
base2.rename(columns = {'Q008':'Q08_Banheiro/Residencia'}, inplace = True)
base2.rename(columns = {'Q009':'Q09_Quartos/Residencia'}, inplace = True)
base2.rename(columns = {'Q010':'Q10_Carro/Residencia'}, inplace = True)
base2.rename(columns = {'Q011':'Q11_Moto/Residencia'}, inplace = True)
base2.rename(columns = {'Q012':'Q12_Geladeira/Residencia'}, inplace = True)
base2.rename(columns = {'Q013':'Q13_Freezer/Residencia'}, inplace = True)
base2.rename(columns = {'Q014':'Q14_Maq_Lavar_Roupa/Residencia'}, inplace = True)
base2.rename(columns = {'Q015':'Q15_Maq_Secar_Roupa/Residencia'}, inplace = True)
base2.rename(columns = {'Q016':'Q16_Micro-ondas/Residencia'}, inplace = True)
base2.rename(columns = {'Q017':'Q17_Maq_Lavar_Louca/Residencia'}, inplace = True)
base2.rename(columns = {'Q018':'Q18_Aspirador/Residencia'}, inplace = True)
base2.rename(columns = {'Q019':'Q19_TV/Residencia'}, inplace = True)
base2.rename(columns = {'Q020':'Q20_DVD/Residencia'}, inplace = True)
base2.rename(columns = {'Q021':'Q21_TV_Assinatura/Residencia'}, inplace = True)
base2.rename(columns = {'Q022':'Q22_Celular/Residencia'}, inplace = True)
base2.rename(columns = {'Q023':'Q23_Telefone_Fixo/Residencia'}, inplace = True)
base2.rename(columns = {'Q024':'Q24_PC/Residencia'}, inplace = True)
base2.rename(columns = {'Q025':'Q25_Acesso_Internet'}, inplace = True)

In [26]:
# Traduz codigos numericos para categorias textuais mais legiveis.
base2['TP_ESCOLA'].replace([1, 2, 3], ['Nao Respondeu', 'Publica', 'Privada'], inplace=True)
base2['TP_DEPENDENCIA_ADM_ESC'].replace([1, 2, 3, 4], ['Federal', 'Estadual', 'Municipal', 'Privada'], inplace=True)

C:\Users\Gabriel\AppData\Local\Temp\ipykernel_4168\2009922221.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  base2['TP_ESCOLA'].replace([1, 2, 3], ['Nao Respondeu', 'Publica', 'Privada'], inplace=True)
C:\Users\Gabriel\AppData\Local\Temp\ipykernel_4168\2009922221.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are set

### Criacao da variavel nota media

criacao da nota com as medias e a exclusao nas colunas de notas

In [27]:
# Consolida as notas em uma metrica unica para simplificar comparacoes posteriores.
base2['NU_MEDIA_GERAL'] = base2[['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_REDACAO','NU_NOTA_MT']].fillna(0).mean(axis=1)

In [28]:
base2.head()

,TP_FAIXA_ETARIA,TP_SEXO,TP_ESCOLA,TP_DEPENDENCIA_ADM_ESC,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,Q01_Educacao_Pai,Q02_Educacao_Mae,Q03_Grupo_Ocupacao_Pai,Q04_Grupo_Ocupacao_Mae,Q05_Moradores/Residencia,Q06_Renda_Familiar_Mensal,Q07_Empregado_Domestico,Q08_Banheiro/Residencia,Q09_Quartos/Residencia,Q10_Carro/Residencia,Q11_Moto/Residencia,Q12_Geladeira/Residencia,Q13_Freezer/Residencia,Q14_Maq_Lavar_Roupa/Residencia,Q15_Maq_Secar_Roupa/Residencia,Q16_Micro-ondas/Residencia,Q17_Maq_Lavar_Louca/Residencia,Q18_Aspirador/Residencia,Q19_TV/Residencia,Q20_DVD/Residencia,Q21_TV_Assinatura/Residencia,Q22_Celular/Residencia,Q23_Telefone_Fixo/Residencia,Q24_PC/Residencia,Q25_Acesso_Internet,NU_MEDIA_GERAL
2,6,F,Nao Respondeu,NaN,502.0,498.9,475.6,363.2,700.0,H,E,C,F,5,C,A,B,D,B,A,B,A,B,A,B,A,A,B,A,A,A,A,A,B,507.94
3,2,F,Publica,Estadual,459.0,508.5,507.2,466.7,880.0,D,D,B,B,5,C,A,B,B,A,A,B,A,A,A,A,A,A,B,A,A,D,A,A,B,564.28
4,3,F,Publica,Estadual,402.5,379.2,446.9,338.3,560.0,B,B,A,A,4,B,A,B,A,A,A,B,A,A,A,A,A,A,B,A,A,B,A,A,A,425.38
9,11,M,Nao Respondeu,NaN,564.7,630.3,610.4,680.2,600.0,H,E,F,D,2,F,A,B,C,B,B,B,B,B,A,B,A,B,C,B,A,C,A,B,B,617.12
10,8,M,Nao Respondeu,NaN,644.9,620.2,626.9,736.3,860.0,F,C,D,B,4,B,A,C,C,A,A,B,A,B,A,B,A,B,B,A,A,E,A,B,B,697.66


In [29]:
# Remove as notas individuais apos gerar a media geral.
base2 = base2.drop(columns=['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_REDACAO','NU_NOTA_MT'])

## Final

In [30]:
import pandas as pd

def comparar_colunas(df1, df2):
    colunas_df1 = set(df1.columns)
    colunas_df2 = set(df2.columns)

    colunas_comuns = sorted(colunas_df1 & colunas_df2)
    colunas_somente_df1 = sorted(colunas_df1 - colunas_df2)
    colunas_somente_df2 = sorted(colunas_df2 - colunas_df1)

    print("✅ COLUNAS PRESENTES NAS DUAS BASES:")
    for c in colunas_comuns:
        print(f" - {c}")

    print("\n❌ COLUNAS REMOVIDAS (SÓ EXISTEM NA PRIMEIRA BASE):")
    for c in colunas_somente_df1:
        print(f" - {c}")
    if not colunas_somente_df1:
        print(" - NENHUMA")

    print("\n📥 COLUNAS NOVAS (SÓ EXISTEM NA SEGUNDA BASE):")
    for c in colunas_somente_df2:
        print(f" - {c}")
    if not colunas_somente_df2:
        print(" - NENHUMA")

    return colunas_comuns, colunas_somente_df1, colunas_somente_df2


# EXEMPLO DE USO
# base_original = pd.read_csv("base_original.csv")
# base_limpa = pd.read_csv("base_limpa.csv")
comum, df1_unicos, df2_unicos = comparar_colunas(base, base2)

✅ COLUNAS PRESENTES NAS DUAS BASES:
 - TP_DEPENDENCIA_ADM_ESC
 - TP_ESCOLA
 - TP_FAIXA_ETARIA
 - TP_SEXO

❌ COLUNAS REMOVIDAS (SÓ EXISTEM NA PRIMEIRA BASE):
 - CO_MUNICIPIO_ESC
 - CO_MUNICIPIO_PROVA
 - CO_PROVA_CH
 - CO_PROVA_CN
 - CO_PROVA_LC
 - CO_PROVA_MT
 - CO_UF_ESC
 - CO_UF_PROVA
 - IN_TREINEIRO
 - NO_MUNICIPIO_ESC
 - NO_MUNICIPIO_PROVA
 - NU_ANO
 - NU_INSCRICAO
 - NU_NOTA_CH
 - NU_NOTA_CN
 - NU_NOTA_COMP1
 - NU_NOTA_COMP2
 - NU_NOTA_COMP3
 - NU_NOTA_COMP4
 - NU_NOTA_COMP5
 - NU_NOTA_LC
 - NU_NOTA_MT
 - NU_NOTA_REDACAO
 - Q001
 - Q002
 - Q003
 - Q004
 - Q005
 - Q006
 - Q007
 - Q008
 - Q009
 - Q010
 - Q011
 - Q012
 - Q013
 - Q014
 - Q015
 - Q016
 - Q017
 - Q018
 - Q019
 - Q020
 - Q021
 - Q022
 - Q023
 - Q024
 - Q025
 - SG_UF_ESC
 - SG_UF_PROVA
 - TP_ANO_CONCLUIU
 - TP_COR_RACA
 - TP_ENSINO
 - TP_ESTADO_CIVIL
 - TP_LINGUA
 - TP_LOCALIZACAO_ESC
 - TP_NACIONALIDADE
 - TP_PRESENCA_CH
 - TP_PRESENCA_CN
 - TP_PRESENCA_LC
 - TP_PRESENCA_MT
 - TP_SIT_FUNC_ESC
 - TP_STATUS_REDACAO
 - TP_ST

In [31]:
final_total_linhas = base2.shape[0]
final_total_colunas = base2.shape[1]
print(f"Comecou: {base.shape[0]} finalizou: {final_total_linhas} retirei: {base.shape[0] - final_total_linhas} - {round((base.shape[0] -final_total_linhas) / base.shape[0],2) }% ")
print(f"Comecou: {base.shape[1]} finalizou: {final_total_colunas} retirei: {base.shape[1] - final_total_colunas} - {round((base.shape[1] -final_total_colunas) / base.shape[1],2)} %")

Comecou: 3933955 finalizou: 2166843 retirei: 1767112 - 0.45% 
Comecou: 76 finalizou: 30 retirei: 46 - 0.61 %


COLUNAS PRESENTES NAS DUAS BASES:
 - TP_DEPENDENCIA_ADM_ESC
 - TP_ESCOLA
 - TP_FAIXA_ETARIA
 - TP_SEXO

❌ COLUNAS REMOVIDAS (SÓ EXISTEM NA PRIMEIRA BASE):
 - CO_MUNICIPIO_ESC - mais de 50% nulos
 - CO_MUNICIPIO_PROVA  - sem utilidade
 - CO_PROVA_CH - sem utilidade
 - CO_PROVA_CN - sem utilidade
 - CO_PROVA_LC - sem utilidade
 - CO_PROVA_MT - sem utilidade
 - CO_UF_ESC - mais de 50% nulos
 - CO_UF_PROVA - sem utilidade
 - IN_TREINEIRO - deixei os que nao foram como treineiros 
 - NO_MUNICIPIO_ESC - mais de 50% nulos
 - NO_MUNICIPIO_PROVA  - sem utilidade
 - NU_ANO - sem utilidade
 - NU_INSCRICAO - sem utilidade
 - NU_NOTA_CH - usado para criar uma nova coluna
 - NU_NOTA_CN - usado para criar uma nova coluna
 - NU_NOTA_COMP1 - sem utilidade
 - NU_NOTA_COMP2 - sem utilidade
 - NU_NOTA_COMP3 - sem utilidade
 - NU_NOTA_COMP4 - sem utilidade
 - NU_NOTA_COMP5 - sem utilidade
 - NU_NOTA_LC - usado para criar uma nova coluna
 - NU_NOTA_MT - usado para criar uma nova coluna
 - NU_NOTA_REDACAO - usado para criar uma nova coluna
 - Q001 - troquei de nome 
 - Q002- troquei de nome 
 - Q003- troquei de nome 
 - Q004 - troquei de nome 
 - Q005 - troquei de nome 
 - Q006 - troquei de nome 
 - Q007 - troquei de nome 
 - Q008 - troquei de nome 
 - Q009 - troquei de nome 
 - Q010 - troquei de nome 
 - Q011 - troquei de nome 
 - Q012 - troquei de nome 
 - Q013 - troquei de nome 
 - Q014 - troquei de nome 
 - Q015 - troquei de nome 
 - Q016 - troquei de nome 
 - Q017 - troquei de nome 
 - Q018 - troquei de nome 
 - Q019 - troquei de nome 
 - Q020 - troquei de nome 
 - Q021 - troquei de nome 
 - Q022 - troquei de nome 
 - Q023 - troquei de nome 
 - Q024 - troquei de nome 
 - Q025 - troquei de nome 
 - SG_UF_ESC - mais de 50% nulos
 - SG_UF_PROVA - sem utilidade
 - TP_ANO_CONCLUIU - sem utilidade
 - TP_COR_RACA - sem utilidade
 - TP_ENSINO - sem utilidade
 - TP_ESTADO_CIVIL - sem utilidade
 - TP_LINGUA - sem utilidade
 - TP_LOCALIZACAO_ESC - mais de 50% nulos
 - TP_NACIONALIDADE - sem utilidade
 - TP_PRESENCA_CH - deixei os que estavam presentes
 - TP_PRESENCA_CN - deixei os que estavam presentes
 - TP_PRESENCA_LC - deixei os que estavam presentes
 - TP_PRESENCA_MT - deixei os que estavam presentes
 - TP_SIT_FUNC_ESC - mais de 50% nulos
 - TP_STATUS_REDACAO - deixei os que estavam presentes
 - TP_ST_CONCLUSAO - deixei os que estavam presentes
 - TX_GABARITO_CH - deixei os que estavam presentes
 - TX_GABARITO_CN - deixei os que estavam presentes
 - TX_GABARITO_LC - deixei os que estavam presentes
 - TX_GABARITO_MT - deixei os que estavam presentes
 - TX_RESPOSTAS_CH - deixei os que estavam presentes
 - TX_RESPOSTAS_CN - deixei os que estavam presentes
 - TX_RESPOSTAS_LC - deixei os que estavam presentes
 - TX_RESPOSTAS_MT - deixei os que estavam presentes

📥 COLUNAS NOVAS (SÓ EXISTEM NA SEGUNDA BASE):
 - NU_MEDIA_GERAL - media do soma das notas(NU_NOTA_CN + NU_NOTA_CH + NU_NOTA_LC + NU_NOTA_REDACAO + NU_NOTA_MT) 
 - Q01_Educacao_Pai - novo nome
 - Q02_Educacao_Mae - novo nome
 - Q03_Grupo_Ocupacao_Pai - novo nome
 - Q04_Grupo_Ocupacao_Mae - novo nome
 - Q05_Moradores/Residencia - novo nome
 - Q06_Renda_Familiar_Mensal - novo nome
 - Q07_Empregado_Domestico - novo nome
 - Q08_Banheiro/Residencia - novo nome
 - Q09_Quartos/Residencia - novo nome
 - Q10_Carro/Residencia - novo nome
 - Q11_Moto/Residencia - novo nome
 - Q12_Geladeira/Residencia - novo nome
 - Q13_Freezer/Residencia - novo nome
 - Q14_Maq_Lavar_Roupa/Residencia - novo nome
 - Q15_Maq_Secar_Roupa/Residencia - novo nome
 - Q16_Micro-ondas/Residencia - novo nome
 - Q17_Maq_Lavar_Louca/Residencia - novo nome
 - Q18_Aspirador/Residencia - novo nome
 - Q19_TV/Residencia - novo nome
 - Q20_DVD/Residencia - novo nome
 - Q21_TV_Assinatura/Residencia - novo nome
 - Q22_Celular/Residencia - novo nome
 - Q23_Telefone_Fixo/Residencia - novo nome
 - Q24_PC/Residencia - novo nome
 - Q25_Acesso_Internet - novo nome

In [32]:
base2['TP_FAIXA_ETARIA'].value_counts().sort_index()

TP_FAIXA_ETARIA
1      15319
2     406881
3     664085
4     288931
5     168604
6     108482
7      77653
8      59628
9      46613
10     36046
11    114191
12     60070
13     44812
14     31508
15     20056
16     12311
17      6957
18      3026
19      1188
20       482
Name: count, dtype: int64

In [33]:
base2.columns

Index(['TP_FAIXA_ETARIA', 'TP_SEXO', 'TP_ESCOLA', 'TP_DEPENDENCIA_ADM_ESC',
       'Q01_Educacao_Pai', 'Q02_Educacao_Mae', 'Q03_Grupo_Ocupacao_Pai',
       'Q04_Grupo_Ocupacao_Mae', 'Q05_Moradores/Residencia',
       'Q06_Renda_Familiar_Mensal', 'Q07_Empregado_Domestico',
       'Q08_Banheiro/Residencia', 'Q09_Quartos/Residencia',
       'Q10_Carro/Residencia', 'Q11_Moto/Residencia',
       'Q12_Geladeira/Residencia', 'Q13_Freezer/Residencia',
       'Q14_Maq_Lavar_Roupa/Residencia', 'Q15_Maq_Secar_Roupa/Residencia',
       'Q16_Micro-ondas/Residencia', 'Q17_Maq_Lavar_Louca/Residencia',
       'Q18_Aspirador/Residencia', 'Q19_TV/Residencia', 'Q20_DVD/Residencia',
       'Q21_TV_Assinatura/Residencia', 'Q22_Celular/Residencia',
       'Q23_Telefone_Fixo/Residencia', 'Q24_PC/Residencia',
       'Q25_Acesso_Internet', 'NU_MEDIA_GERAL'],
      dtype='object')

In [34]:
base2['TP_DEPENDENCIA_ADM_ESC'].count()

np.int64(721429)

In [35]:
base2['TP_DEPENDENCIA_ADM_ESC'].value_counts(dropna=False).sort_index()

TP_DEPENDENCIA_ADM_ESC
Estadual      456911
Federal        43946
Municipal       5832
Privada       214740
NaN          1445414
Name: count, dtype: int64

In [36]:
soma_D= base2['TP_DEPENDENCIA_ADM_ESC'].isin(['Federal', 'Estadual', 'Municipal', 'Privada']).sum()
soma_D

np.int64(721429)

In [37]:
base2['TP_ESCOLA'].count()

np.int64(2166843)

In [38]:
base2['TP_ESCOLA'].value_counts(dropna=False).sort_index()

TP_ESCOLA
Nao Respondeu    1115985
Privada           221328
Publica           829530
Name: count, dtype: int64

In [39]:
soma_E = base2['TP_ESCOLA'].isin(['Publica', 'Privada']).sum()
soma_E

np.int64(1050858)

In [40]:
# === CRIAR UMA BASE SÓ COM NULOS ===
# Separa primeiro os registros sem classificacao de tipo de escola.
base_nulo = base2[base2['TP_ESCOLA'].isna()]

# === CRIAR BASES PARA CADA CATEGORIA ===
bases_por_tipo = {}
for tipo in sorted(base2['TP_ESCOLA'].dropna().unique()):
    nome_limpo = str(tipo).strip().replace(" ", "_").replace("/", "_")  # remove espaços e barras
    bases_por_tipo[nome_limpo] = base2[base2['TP_ESCOLA'] == tipo]

# === SALVAR AS BASES (SEM AS COLUNAS 'TP_ESCOLA' E 'TP_DEPENDENCIA_ADM_ESC') ===
for tipo, df in bases_por_tipo.items():
    df.drop(columns=['TP_ESCOLA', 'TP_DEPENDENCIA_ADM_ESC'], errors="ignore") \
      .to_csv(f"bases_limpas/base_dependencia_{tipo}.csv", sep=";", index=False, encoding="utf-8")

# SALVAR BASE COM NULOS
base_nulo.drop(columns=['TP_ESCOLA', 'TP_DEPENDENCIA_ADM_ESC'], errors="ignore") \
    .to_csv("bases_limpas/base_dependencia_nulo.csv", sep=";", index=False, encoding="utf-8")

print("✅ BASES SALVAS COM SUCESSO! (SEM AS COLUNAS 'TP_ESCOLA' E 'TP_DEPENDENCIA_ADM_ESC')")

✅ BASES SALVAS COM SUCESSO! (SEM AS COLUNAS 'TP_ESCOLA' E 'TP_DEPENDENCIA_ADM_ESC')


As bases finais foram separadas por categoria de `TP_ESCOLA`, com um arquivo adicional reservado para registros com valor nulo nessa variavel.

In [41]:
# === BASE COM NULOS ===
base_nulo = base2[base2['TP_ESCOLA'].isna()].copy()

# === PEGAR OS VALORES ÚNICOS VÁLIDOS ===
valores = sorted(base2['TP_ESCOLA'].dropna().unique())

# === CRIAR NOMES SEGUROS PARA CADA BASE ===
for tipo in valores:
    nome_var = f"base_{str(tipo).strip().replace(' ', '_')}"   # remove espaços
    globals()[nome_var] = base2[base2['TP_ESCOLA'] == tipo].copy()
    print(f"{nome_var} → {len(globals()[nome_var])} registros")

# TAMANHO DA BASE NULA
print(f"base_nulo → {len(base_nulo)} registros")

base_Nao_Respondeu → 1115985 registros
base_Privada → 221328 registros
base_Publica → 829530 registros
base_nulo → 0 registros


In [42]:
base_Privada.info()

<class 'pandas.core.frame.DataFrame'>
Index: 221328 entries, 122 to 3933953
Data columns (total 30 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   TP_FAIXA_ETARIA                 221328 non-null  int64  
 1   TP_SEXO                         221328 non-null  object 
 2   TP_ESCOLA                       221328 non-null  object 
 3   TP_DEPENDENCIA_ADM_ESC          194237 non-null  object 
 4   Q01_Educacao_Pai                221328 non-null  object 
 5   Q02_Educacao_Mae                221328 non-null  object 
 6   Q03_Grupo_Ocupacao_Pai          221328 non-null  object 
 7   Q04_Grupo_Ocupacao_Mae          221328 non-null  object 
 8   Q05_Moradores/Residencia        221328 non-null  int64  
 9   Q06_Renda_Familiar_Mensal       221328 non-null  object 
 10  Q07_Empregado_Domestico         221328 non-null  object 
 11  Q08_Banheiro/Residencia         221328 non-null  object 
 12  Q09_Quartos/Reside

In [43]:
base_Publica.info()

<class 'pandas.core.frame.DataFrame'>
Index: 829530 entries, 3 to 3933948
Data columns (total 30 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   TP_FAIXA_ETARIA                 829530 non-null  int64  
 1   TP_SEXO                         829530 non-null  object 
 2   TP_ESCOLA                       829530 non-null  object 
 3   TP_DEPENDENCIA_ADM_ESC          527190 non-null  object 
 4   Q01_Educacao_Pai                829530 non-null  object 
 5   Q02_Educacao_Mae                829530 non-null  object 
 6   Q03_Grupo_Ocupacao_Pai          829530 non-null  object 
 7   Q04_Grupo_Ocupacao_Mae          829530 non-null  object 
 8   Q05_Moradores/Residencia        829530 non-null  int64  
 9   Q06_Renda_Familiar_Mensal       829530 non-null  object 
 10  Q07_Empregado_Domestico         829530 non-null  object 
 11  Q08_Banheiro/Residencia         829530 non-null  object 
 12  Q09_Quartos/Residenc

In [44]:
base_Nao_Respondeu.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1115985 entries, 2 to 3933950
Data columns (total 30 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   TP_FAIXA_ETARIA                 1115985 non-null  int64  
 1   TP_SEXO                         1115985 non-null  object 
 2   TP_ESCOLA                       1115985 non-null  object 
 3   TP_DEPENDENCIA_ADM_ESC          2 non-null        object 
 4   Q01_Educacao_Pai                1115985 non-null  object 
 5   Q02_Educacao_Mae                1115985 non-null  object 
 6   Q03_Grupo_Ocupacao_Pai          1115985 non-null  object 
 7   Q04_Grupo_Ocupacao_Mae          1115985 non-null  object 
 8   Q05_Moradores/Residencia        1115985 non-null  int64  
 9   Q06_Renda_Familiar_Mensal       1115985 non-null  object 
 10  Q07_Empregado_Domestico         1115985 non-null  object 
 11  Q08_Banheiro/Residencia         1115985 non-null  object 
 12  Q09_Q

In [45]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2166843 entries, 2 to 3933953
Data columns (total 30 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   TP_FAIXA_ETARIA                 int64  
 1   TP_SEXO                         object 
 2   TP_ESCOLA                       object 
 3   TP_DEPENDENCIA_ADM_ESC          object 
 4   Q01_Educacao_Pai                object 
 5   Q02_Educacao_Mae                object 
 6   Q03_Grupo_Ocupacao_Pai          object 
 7   Q04_Grupo_Ocupacao_Mae          object 
 8   Q05_Moradores/Residencia        int64  
 9   Q06_Renda_Familiar_Mensal       object 
 10  Q07_Empregado_Domestico         object 
 11  Q08_Banheiro/Residencia         object 
 12  Q09_Quartos/Residencia          object 
 13  Q10_Carro/Residencia            object 
 14  Q11_Moto/Residencia             object 
 15  Q12_Geladeira/Residencia        object 
 16  Q13_Freezer/Residencia          object 
 17  Q14_Maq_Lavar_Roupa/Residencia  

In [46]:
base2.columns

Index(['TP_FAIXA_ETARIA', 'TP_SEXO', 'TP_ESCOLA', 'TP_DEPENDENCIA_ADM_ESC',
       'Q01_Educacao_Pai', 'Q02_Educacao_Mae', 'Q03_Grupo_Ocupacao_Pai',
       'Q04_Grupo_Ocupacao_Mae', 'Q05_Moradores/Residencia',
       'Q06_Renda_Familiar_Mensal', 'Q07_Empregado_Domestico',
       'Q08_Banheiro/Residencia', 'Q09_Quartos/Residencia',
       'Q10_Carro/Residencia', 'Q11_Moto/Residencia',
       'Q12_Geladeira/Residencia', 'Q13_Freezer/Residencia',
       'Q14_Maq_Lavar_Roupa/Residencia', 'Q15_Maq_Secar_Roupa/Residencia',
       'Q16_Micro-ondas/Residencia', 'Q17_Maq_Lavar_Louca/Residencia',
       'Q18_Aspirador/Residencia', 'Q19_TV/Residencia', 'Q20_DVD/Residencia',
       'Q21_TV_Assinatura/Residencia', 'Q22_Celular/Residencia',
       'Q23_Telefone_Fixo/Residencia', 'Q24_PC/Residencia',
       'Q25_Acesso_Internet', 'NU_MEDIA_GERAL'],
      dtype='object')